In [1]:
import pandas as pd

listings = pd.read_csv("data/listings_clean.csv")
numbers = pd.Series([1, 2, 3, 4, 5])
print(numbers.mean())
print(numbers.median())


3.0
3.0


In [2]:
reviews = listings['number_of_reviews']
print(reviews.mean())
print(reviews.median())

35.75404459991255
9.0


In [3]:
listings["number_of_reviews"].describe()

count    4574.000000
mean       35.754045
std        78.377200
min         0.000000
25%         2.000000
50%         9.000000
75%        29.000000
max      1293.000000
Name: number_of_reviews, dtype: float64

In [5]:
reviews = listings["number_of_reviews"]
print(reviews.describe())
print(reviews.quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99]))
q1 = reviews.quantile(0.25)
q3 = reviews.quantile(0.75)
iqr = q3 - q1
print(iqr)

count    4574.000000
mean       35.754045
std        78.377200
min         0.000000
25%         2.000000
50%         9.000000
75%        29.000000
max      1293.000000
Name: number_of_reviews, dtype: float64
0.10      0.00
0.25      2.00
0.50      9.00
0.75     29.00
0.90     94.00
0.99    400.27
Name: number_of_reviews, dtype: float64
27.0


In [6]:
upper = q3 + 1.5 * iqr
lower = q1 - 1.5 * iqr
print(f"Upper bound: {upper}, Lower bound: {lower}")

Upper bound: 69.5, Lower bound: -38.5


In [7]:
reviews = listings["number_of_reviews"]

listings["reviews_outlier"] = (reviews < lower) | (reviews > upper)
print("Outlier listings:", listings["reviews_outlier"].sum())

reviews_without_outliers = listings.loc[
    ~listings["reviews_outlier"], "number_of_reviews"
]

stats = pd.DataFrame({
    "All listings": reviews.describe()[["count", "mean", "50%", "std"]],
    "Without outliers": reviews_without_outliers.describe()[
        ["count", "mean", "50%", "std"]
    ],
}).rename(index={"50%": "median"})

print(stats)

percent_change = (
    (stats["Without outliers"] - stats["All listings"])
    / stats["All listings"]
    * 100
)
print("Percent change (%):")
print(percent_change)

Outlier listings: 582
        All listings  Without outliers
count    4574.000000       3992.000000
mean       35.754045         12.947395
median      9.000000          7.000000
std        78.377200         15.537791
Percent change (%):
count    -12.724093
mean     -63.787608
median   -22.222222
std      -80.175624
dtype: float64


In [8]:
listings.groupby("room_type")["number_of_reviews"].describe().round(0)

,count,mean,std,min,25%,50%,75%,max
room_type,,,,,,,,
Entire home/apt,3760.0,34.0,76.0,0.0,2.0,8.0,27.0,1293.0
Hotel room,3.0,61.0,16.0,49.0,52.0,56.0,68.0,79.0
Private room,787.0,41.0,84.0,0.0,2.0,12.0,38.0,867.0
Shared room,24.0,114.0,122.0,0.0,28.0,82.0,146.0,510.0


In [9]:
correlation_columns = [
    "number_of_reviews",
    "price",
    "accommodates",
    "bedrooms",
    "availability_365",
    "review_scores_rating",
    "minimum_nights",
    "days_since_review",
]

correlations = listings[correlation_columns].corr()["number_of_reviews"].drop(
    "number_of_reviews"
)
print(correlations)

price                  -0.091789
accommodates           -0.050848
bedrooms               -0.144178
availability_365        0.183156
review_scores_rating   -0.039295
minimum_nights         -0.065609
days_since_review      -0.185301
Name: number_of_reviews, dtype: float64


In [10]:
corr_all = listings["number_of_reviews"].corr(listings["availability_365"])

corr_without_outliers = listings.loc[
    ~listings["reviews_outlier"], "number_of_reviews"
].corr(
    listings.loc[~listings["reviews_outlier"], "availability_365"]
)

print("With outliers:", corr_all)
print("Without outliers:", corr_without_outliers)

With outliers: 0.18315645212034498
Without outliers: 0.002236567165322214
